# Exercise Sheet 3 — Singular value decomposition (SVD)
**Course:** Big Data Science in Finance (WI3436TU)
**Data:** `E3/mona_lisa_public_domain.jpg`, `E3/market_data_2007_2025.csv`
(SPY, QQQ, TLT, HYG, VIX)

This notebook works through Problems 1-3 of Exercise Sheet 3. Code cells compute the required
quantities and produce the plots; markdown cells contain the hand calculations, proofs and
written discussion. The two data files are expected in the folder `E3/` next to this notebook.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option("display.float_format", lambda x: f"{x:0.6f}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

DATA_DIR = Path("E3")
IMAGE_PATH = DATA_DIR / "mona_lisa_public_domain.jpg"
MARKET_PATH = DATA_DIR / "market_data_2007_2025.csv"
ASSETS = ["SPY", "QQQ", "TLT", "HYG", "VIX"]

### Notation

For $\mathbf X \in \mathbb R^{n \times p}$, write its SVD using the $r$ nonzero singular values as
$$\mathbf X = \sum_{j=1}^r \sigma_j \mathbf u_j \mathbf v_j^\top, \qquad \sigma_1 \ge \dots \ge \sigma_r > 0.$$

Unless stated otherwise, the SVD is computed with `np.linalg.svd(X, full_matrices=False)`.

In [ ]:
def rank_k_approx(U, s, Vt, k):
    """Truncated SVD X_k = sum_{j<=k} sigma_j u_j v_j^T."""
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]


def energy_share(s):
    """Cumulative squared-energy share sum_{j<=k} sigma_j^2 / sum_j sigma_j^2, for k = 1, 2, ..."""
    return np.cumsum(s**2) / np.sum(s**2)

## Problem 1

### a)

Let $\mathbf A = \begin{pmatrix} 3 & 0 \\ 4 & 0 \end{pmatrix}$. Find the rank-one SVD by hand.
Verify $\mathbf A\mathbf v_1 = \sigma_1 \mathbf u_1$ and $\mathbf A^\top \mathbf u_1 = \sigma_1 \mathbf v_1$.

**Hand calculation.**

In [ ]:
# numerical check of the hand calculation
A = np.array([[3.0, 0.0],
              [4.0, 0.0]])
U_A, s_A, Vt_A = np.linalg.svd(A, full_matrices=False)
u1, sigma1, v1 = U_A[:, 0], s_A[0], Vt_A[0]

print("singular values:", s_A)
print("u1 =", u1, "  v1 =", v1)
print("A v1       =", A @ v1, "  sigma1 u1 =", sigma1 * u1)
print("A^T u1     =", A.T @ u1, "  sigma1 v1 =", sigma1 * v1)

### b)

Let $\mathbf B = \begin{pmatrix} 0 & 2 \\ 0 & 0 \end{pmatrix}$. Compute its singular values. Use
this example to explain why the spectral norm of a nonsymmetric matrix is its largest singular
value.

In [ ]:
B = np.array([[0.0, 2.0],
              [0.0, 0.0]])

print("singular values of B:", np.linalg.svd(B, compute_uv=False))
print("eigenvalues of B:    ", np.linalg.eigvals(B))
print("spectral norm ||B||_2 =", np.linalg.norm(B, 2))
print("B @ (0, 1) =", B @ np.array([0.0, 1.0]))

**Discussion.**

### c)

Starting from $\mathbf C = \mathbf U \boldsymbol\Sigma \mathbf V^\top$ and orthogonal invariance,
show that $\|\mathbf C\|_2 = \sigma_1(\mathbf C)$, and $\|\mathbf C\|_F^2 = \sum_j \sigma_j^2(\mathbf C)$.

**Proof.**

### d)

A matrix has nonzero singular values $8, 4, 1, 1$. For $k = 2$, compute
$\|\mathbf X - \mathbf X_k\|_F$, the retained squared-energy share, and the relative Frobenius
error. State what the Eckart–Young–Mirsky theorem says about this error.

In [ ]:
s = np.array([8.0, 4.0, 1.0, 1.0])
k = 2

error_F = np.sqrt(np.sum(s[k:]**2))
retained_share = np.sum(s[:k]**2) / np.sum(s**2)
relative_error = error_F / np.sqrt(np.sum(s**2))

print(f"||X - X_2||_F          = {error_F:.4f}")
print(f"retained energy share  = {retained_share:.4f}")
print(f"relative Frobenius err = {relative_error:.4f}")

**Discussion.**

## Problem 2

The file `E3/mona_lisa_public_domain.jpg` is a public domain image of *Mona Lisa*. Load it with
`Pillow`, convert it to grayscale, resize it to height 360 pixels while preserving its aspect
ratio, scale its intensities to $[0, 1]$. Let the resulting matrix be $\mathbf X \in \mathbb R^{n \times p}$.

In [ ]:
img = Image.open(IMAGE_PATH).convert("L")   # "L" = grayscale

new_height = 360
width, height = img.size
new_width = round(width * new_height / height)   # keep the aspect ratio
img = img.resize((new_width, new_height))

X = np.asarray(img, dtype=float) / 255.0          # intensities in [0, 1]

### a)

Compute $n$, $p$, $\operatorname{rank}(\mathbf X)$ numerically, and the intensity range. Display
the image with a grayscale color map.

In [ ]:
n, p = X.shape
print(f"n = {n}, p = {p}")
print(f"numerical rank(X) = {np.linalg.matrix_rank(X)}")
print(f"intensity range = [{X.min():.4f}, {X.max():.4f}]")

plt.figure(figsize=(4, 6))
plt.imshow(X, cmap="gray", vmin=0, vmax=1)
plt.title("Grayscale image X")
plt.axis("off")
plt.show()

### b)

Compute its SVD with `full_matrices=False`. Check numerically that the following ones are small:
$$\left\|\mathbf U^\top\mathbf U - \mathbf I\right\|_2, \qquad \left\|\mathbf V^\top\mathbf V - \mathbf I\right\|_2, \qquad \frac{\left\|\mathbf X - \mathbf U\boldsymbol\Sigma\mathbf V^\top\right\|_F}{\|\mathbf X\|_F}.$$

In [ ]:
U, s, Vt = np.linalg.svd(X, full_matrices=False)
V = Vt.T
I = np.eye(len(s))

print("U shape:", U.shape, " s shape:", s.shape, " Vt shape:", Vt.shape)
print(f"||U^T U - I||_2              = {np.linalg.norm(U.T @ U - I, 2):.2e}")
print(f"||V^T V - I||_2              = {np.linalg.norm(V.T @ V - I, 2):.2e}")
rel = np.linalg.norm(X - U @ np.diag(s) @ Vt, "fro") / np.linalg.norm(X, "fro")
print(f"||X - U S V^T||_F / ||X||_F  = {rel:.2e}")

### c)

Reconstruct and display $\mathbf X_k$ for $k \in \{1, 5, 10, 20, 50, 100\}$. Clip displayed
intensities to $[0, 1]$, but compute all errors before clipping.

In [ ]:
k_values = [1, 5, 10, 20, 50, 100]
X_k = {k: rank_k_approx(U, s, Vt, k) for k in k_values}   # NOT clipped

fig, axes = plt.subplots(1, len(k_values), figsize=(18, 5))
for ax, k in zip(axes, k_values):
    ax.imshow(np.clip(X_k[k], 0, 1), cmap="gray", vmin=0, vmax=1)   # clip for display only
    ax.set_title(f"k = {k}")
    ax.axis("off")
fig.tight_layout()
plt.show()

### d)

For the same values of $k$, compute the actual Frobenius errors and compare them with
$\left(\sum_{j>k}\sigma_j^2\right)^{1/2}$. Explain any discrepancy beyond floating point rounding.

In [ ]:
rows = []
for k in k_values:
    actual = np.linalg.norm(X - X_k[k], "fro")
    predicted = np.sqrt(np.sum(s[k:]**2))
    rows.append({"k": k, "actual ||X - X_k||_F": actual,
                 "sqrt(sum_{j>k} sigma_j^2)": predicted,
                 "difference": actual - predicted})

pd.DataFrame(rows).set_index("k")

**Discussion.**

### e)

Find the smallest $k$ that retains at least 90%, 95%, and 99% of the squared energy. For each
value, compute the relative Frobenius error. Compare the mathematical criteria with your visual
judgment.

In [ ]:
share = energy_share(s)
thresholds = [0.90, 0.95, 0.99]

rows = []
for thr in thresholds:
    k = int(np.argmax(share >= thr)) + 1   # first k with share >= threshold
    Xk = rank_k_approx(U, s, Vt, k)
    rel_err = np.linalg.norm(X - Xk, "fro") / np.linalg.norm(X, "fro")
    rows.append({"threshold": thr, "smallest k": k,
                 "retained share": share[k - 1], "relative Frobenius error": rel_err})

energy_table = pd.DataFrame(rows).set_index("threshold")
energy_table

In [ ]:
fig, axes = plt.subplots(1, len(thresholds) + 1, figsize=(14, 5))
axes[0].imshow(X, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("original")
for ax, thr in zip(axes[1:], thresholds):
    k = energy_table.loc[thr, "smallest k"]
    ax.imshow(np.clip(rank_k_approx(U, s, Vt, k), 0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{thr:.0%} energy: k = {k}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

**Discussion.**

## Problem 3

The file `E3/market_data_2007_2025.csv` contains daily adjusted prices downloaded with
`start_date`= 2007-04-11 and `end_date`= 2026-01-01 for SPY (S&P 500 equity market), QQQ (growth
equities), TLT (long duration US Treasuries), HYG (high yield corporate credit), and VIX
(option-implied S&P 500 volatility index). Use trading days only. Compute the five daily log
changes $x^{(j)}_t = \log P^{(j)}_t - \log P^{(j)}_{t-1}$, and align them by date. Store the
financial series in rows and dates in columns, as in Lecture 3: $\mathbf R \in \mathbb R^{5 \times T}$,
$R_{jt} = x^{(j)}_t$, where $T$ is the number of aligned trading dates. The five rows are SPY,
QQQ, TLT, HYG, and VIX, in that order. The CSV has dates in rows: transpose it before computing
the SVD.

In [ ]:
prices = pd.read_csv(MARKET_PATH, index_col=0, parse_dates=True).sort_index()
prices.columns = [c.replace("^", "") for c in prices.columns]   # e.g. "^VIX" -> "VIX"
prices = prices[ASSETS]

# daily log changes per series (trading days only, missing prices dropped), then align by date
log_changes = pd.concat(
    {a: np.log(prices[a].dropna()).diff().dropna() for a in ASSETS},
    axis=1, join="inner",
)

R = log_changes.T.to_numpy()    # 5 x T: series in rows, dates in columns
dates = log_changes.index
T = R.shape[1]

print("R shape:", R.shape)
print("first date:", dates[0].date(), " last date:", dates[-1].date())
log_changes.head()

### a)

Compute the SVD $\mathbf R = \mathbf U\boldsymbol\Sigma\mathbf V^\top$. State what the left singular
vector $\mathbf u_1 \in \mathbb R^5$ and the right singular vector $\mathbf v_1 \in \mathbb R^T$
represent. Which of $\mathbf v_1$ and $\sigma_1\mathbf v_1$ gives the actual dated coefficients of
the first asset pattern?

In [ ]:
U, s, Vt = np.linalg.svd(R, full_matrices=False)

print("U shape:", U.shape, " s shape:", s.shape, " Vt shape:", Vt.shape)
print("singular values:", np.round(s, 4))
print("u1 =", pd.Series(U[:, 0], index=ASSETS).round(4).to_dict())
print("first five entries of v1:         ", np.round(Vt[0, :5], 5))
print("first five entries of sigma1 * v1:", np.round(s[0] * Vt[0, :5], 5))

**Discussion.**

### b)

Plot the singular values and cumulative squared-energy shares for $\mathbf R$. Treat the
singular-value curve as a scree plot. State whether it has a convincing elbow and give a
defensible elbow-based $k$, if one exists. For use in part c), orient the leading left singular
vector so that its SPY coefficient is positive, and report its coefficients. If you reverse its
sign, also reverse the sign of the paired right singular vector. For the first date, calculate
the rank-one reconstructed column $\sigma_1(\mathbf v_1)_1\mathbf u_1$ and interpret its entries.

In [ ]:
share = energy_share(s)
ks = np.arange(1, len(s) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, s, "o-")
axes[0].set_title("Scree plot: singular values of R")
axes[0].set_xlabel("k")
axes[0].set_ylabel("sigma_k")
axes[1].plot(ks, share, "o-", color="tab:orange")
axes[1].axhline(0.9, color="gray", ls="--", label="90%")
axes[1].set_title("Cumulative squared-energy share")
axes[1].set_xlabel("k")
axes[1].set_ylim(0, 1.05)
axes[1].legend()
for ax in axes:
    ax.set_xticks(ks)
fig.tight_layout()
plt.show()

pd.DataFrame({"sigma_k": s, "energy share": s**2 / np.sum(s**2), "cumulative share": share}, index=ks)

In [ ]:
# orient u1 so that its SPY coefficient (row 0) is positive; flip v1 together with it
if U[0, 0] < 0:
    U[:, 0] *= -1
    Vt[0, :] *= -1

u1 = U[:, 0]
v1 = Vt[0, :]
print("oriented u1:")
print(pd.Series(u1, index=ASSETS).round(4))

# rank-one reconstruction of the first date's column
first_col = s[0] * v1[0] * u1
pd.DataFrame({"rank-one reconstruction": first_col, "actual R[:, 0]": R[:, 0]}, index=ASSETS)

**Discussion.**

### c)

Let $\mathbf u_1$ be the oriented leading left singular vector from part c), and let
$\mathcal T = \{\text{SPY}, \text{QQQ}, \text{TLT}, \text{HYG}\}$ and
$\mathcal I_+ = \{j \in \mathcal T : u_{j1} > 0\}$. For $j \in \mathcal I_+$, set
$w_j = \frac{u_{j1}}{\sum_{\ell \in \mathcal I_+} u_{\ell 1}}$. Compute $\mathcal I_+$ and the weights.
Convert the log changes to simple returns $q^{(j)}_t = \exp(x^{(j)}_t) - 1$. Plot the cumulative return
$$\prod_{s \le t}\left(1 + \sum_{j \in \mathcal I_+} w_j q^{(j)}_s\right) - 1$$
beside that of a daily equally weighted portfolio of the same ETFs.

In [ ]:
etfs = ["SPY", "QQQ", "TLT", "HYG"]
u1_series = pd.Series(u1, index=ASSETS)

I_plus = [j for j in etfs if u1_series[j] > 0]
w = u1_series[I_plus] / u1_series[I_plus].sum()
print("I_+ =", I_plus)
print("weights:")
print(w.round(4))

q = np.exp(log_changes[I_plus]) - 1    # simple returns, dates x assets

svd_port = q @ w                       # daily return of the SVD-weighted portfolio
ew_port = q.mean(axis=1)               # daily equally weighted portfolio of the same ETFs

cum_svd = (1 + svd_port).cumprod() - 1
cum_ew = (1 + ew_port).cumprod() - 1

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
axes[0].plot(cum_svd)
axes[0].set_title("SVD-weighted portfolio")
axes[1].plot(cum_ew, color="tab:orange")
axes[1].set_title("Equally weighted portfolio")
for ax in axes:
    ax.set_ylabel("cumulative return")
fig.tight_layout()
plt.show()

print(f"terminal cumulative return: SVD-weighted = {cum_svd.iloc[-1]:.2%}, "
      f"equally weighted = {cum_ew.iloc[-1]:.2%}")

### d)

For $k = 1, 2, 3, 4$, reconstruct $\mathbf R_k$. Which rank retains at least 90% of the squared
energy?

In [ ]:
rows = []
for k in [1, 2, 3, 4]:
    Rk = rank_k_approx(U, s, Vt, k)
    rows.append({
        "k": k,
        "retained share": share[k - 1],
        "relative Frobenius error": np.linalg.norm(R - Rk, "fro") / np.linalg.norm(R, "fro"),
    })

rank_table = pd.DataFrame(rows).set_index("k")
print("smallest k with at least 90% of the squared energy:",
      rank_table.index[rank_table["retained share"] >= 0.9][0])
rank_table

### e)

For $k = 2$, plot the original changes and their reconstruction over the final 250 trading days.
Plot the residuals separately. Describe what is retained and what is lost.

In [ ]:
R2 = rank_k_approx(U, s, Vt, 2)
last = slice(T - 250, T)
last_dates = dates[last]

fig, axes = plt.subplots(len(ASSETS), 1, figsize=(12, 12), sharex=True)
for i, (ax, asset) in enumerate(zip(axes, ASSETS)):
    ax.plot(last_dates, R[i, last], lw=1, label="original")
    ax.plot(last_dates, R2[i, last], lw=1, label="rank-2 reconstruction")
    ax.set_ylabel(asset)
axes[0].legend(loc="upper right")
axes[0].set_title("Original log changes vs rank-2 reconstruction (final 250 trading days)")
fig.tight_layout()
plt.show()

residuals = R - R2
fig, axes = plt.subplots(len(ASSETS), 1, figsize=(12, 12), sharex=True)
for i, (ax, asset) in enumerate(zip(axes, ASSETS)):
    ax.plot(last_dates, residuals[i, last], lw=1, color="tab:red")
    ax.set_ylabel(asset)
axes[0].set_title("Residuals R - R_2 (final 250 trading days)")
fig.tight_layout()
plt.show()

**Discussion.**

### f)

The portfolio weights in part c) were fitted using the same dates on which performance is
plotted. Explain why neither a large singular value nor a higher terminal cumulative return
establishes a profitable future strategy.

**Discussion.**